In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras import Sequential

In [2]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [3]:
base_model.trainable = False

In [4]:
model_transfer = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(10, activation="softmax")
])

In [5]:
import tensorflow as tf
import numpy as np

(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.cifar10.load_data()
)

print(X_train.shape)
print(X_test.shape)

(50000, 32, 32, 3)
(10000, 32, 32, 3)


In [6]:
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

In [7]:
X_train = tf.image.resize(X_train, (96, 96))
X_test = tf.image.resize(X_test, (96, 96))

In [8]:
X_train = tf.keras.applications.mobilenet_v2.preprocess_input(
    X_train
)

X_test = tf.keras.applications.mobilenet_v2.preprocess_input(
    X_test
)

In [9]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

base_model.trainable = False

In [10]:
model_transfer = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [11]:
model_transfer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
history_transfer = model_transfer.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 214s 294ms/step - accuracy: 0.8224 - loss: 0.5270 - val_accuracy: 0.8600 - val_loss: 0.4079
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 195s 277ms/step - accuracy: 0.8728 - loss: 0.3668 - val_accuracy: 0.8678 - val_loss: 0.3804
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 195s 277ms/step - accuracy: 0.8918 - loss: 0.3109 - val_accuracy: 0.8638 - val_loss: 0.3955
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 195s 276ms/step - accuracy: 0.9061 - loss: 0.2675 - val_accuracy: 0.8684 - val_loss: 0.3817
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 194s 275ms/step - accuracy: 0.9215 - loss: 0.2257 - val_accuracy: 0.8630 - val_loss: 0.4201


In [13]:
test_loss, test_accuracy = model_transfer.evaluate(
    X_test,
    y_test,
    batch_size=64
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

157/157 ━━━━━━━━━━━━━━━━━━━━ 43s 272ms/step - accuracy: 0.8577 - loss: 0.4367
Test Loss: 0.43672290444374084
Test Accuracy: 0.857699990272522


In [14]:
# Unfreeze the base model
base_model.trainable = True

# Freeze most layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Keep the last 20 layers trainable
for layer in base_model.layers[-20:]:
    layer.trainable = True

In [15]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [16]:
history_finetune = model_transfer.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 250s 340ms/step - accuracy: 0.8357 - loss: 0.5324 - val_accuracy: 0.8660 - val_loss: 0.4602
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 227s 322ms/step - accuracy: 0.8872 - loss: 0.3319 - val_accuracy: 0.8688 - val_loss: 0.4255
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 241s 343ms/step - accuracy: 0.9067 - loss: 0.2652 - val_accuracy: 0.8692 - val_loss: 0.4216
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 253s 359ms/step - accuracy: 0.9227 - loss: 0.2234 - val_accuracy: 0.8692 - val_loss: 0.4143
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 254s 360ms/step - accuracy: 0.9339 - loss: 0.1876 - val_accuracy: 0.8736 - val_loss: 0.4106


In [17]:
test_loss_ft, test_accuracy_ft = model_transfer.evaluate(
    X_test,
    y_test,
    batch_size=64
)

print("Fine-Tuned Test Loss:", test_loss_ft)
print("Fine-Tuned Test Accuracy:", test_accuracy_ft)

157/157 ━━━━━━━━━━━━━━━━━━━━ 42s 268ms/step - accuracy: 0.8732 - loss: 0.4115
Fine-Tuned Test Loss: 0.411456823348999
Fine-Tuned Test Accuracy: 0.873199999332428


In [18]:
print("Feature Extraction Accuracy: 86.04%")
print("Fine-Tuning Accuracy:", test_accuracy_ft * 100)

Feature Extraction Accuracy: 86.04%
Fine-Tuning Accuracy: 87.3199999332428
